In [49]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Circle, FancyBboxPatch, Arc, Rectangle
from IPython.display import HTML
from puck2vec import Puck2Vec
from puck2vec_dataset import SeqDataset, PLAYERS_FRAME, FRAME_HZ, MOVE_BINS, MAX_MOVE, discretize_move
plt.rcParams["animation.embed_limit"] = 100

In [ ]:
df = pd.read_csv("data/processed_ppt.csv")
meta = pd.read_csv("data/metadata.csv").set_index("playerId")
ckpt = torch.load("puck2vec.pth", map_location="cpu", weights_only=False)
model = ckpt.eval()


In [ ]:
ARROW_SECONDS = 0.5  # arrows point to where the model expects the player to be this far ahead

def load_shot(game_id, event_id):
    rows = df[df["shot_id"] == f"{game_id}_{event_id}"].reset_index(drop=True)
    if rows.empty:
        raise ValueError(f"no tracking data for game {game_id} event {event_id}")
    shot = SeqDataset(rows, n_frames=model.n_frames)
    if not shot.sequences:
        raise ValueError("no run of valid frames long enough for the model")
    seq = shot.sequences[-1]  # the valid run closest to the shot
    local2pid = {i: pid for pid, i in shot.player_index.items()}
    pids = [local2pid[int(i)] for i in seq["player_ids"]]
    px = rows["puck_x"].to_numpy(np.float32)
    T = len(seq["x"])
    start = next(s for s in range(len(rows) - T + 1) if np.array_equal(px[s:s + T], seq["puck_x"].numpy()))
    sweater = {}
    for p in [f"{side}_on_ice_{i}" for side in ("home", "away") for i in range(1, 7)]:
        for pid, num in zip(rows[f"{p}_id"], rows[f"{p}_sweater_number"]):
            if pd.notna(pid) and pd.notna(num):
                sweater[int(pid)] = int(num)
    keep = slice(len(seq["x"]) - model.n_frames - 1, None)  # last n_frames frames, plus one more for their moves
    seq = {**seq, **{k: seq[k][keep] for k in ("x", "y", "off", "puck_x", "puck_y")}}
    start, T = start + keep.start, model.n_frames + 1
    return {**seq, "pids": pids,
            "sweater": [sweater.get(p, "") for p in pids], "names": [meta["name"].get(p, str(p)) for p in pids],
            "t": -rows["time_to_shot"].to_numpy()[start:start + T], "rows": rows.iloc[start:start + T],
            "xG": rows["shot_xG"].iloc[0], "goal": bool(rows["is_goal"].iloc[0])}

def run_model(seq, target):
    n, K = model.n_frames, PLAYERS_FRAME
    if target == "puck":
        k_t = K
    elif target in seq["pids"]:
        k_t = seq["pids"].index(target)
    elif target != "epv":
        raise ValueError(f"target must be 'puck', 'epv' or a player on the ice for this shot: {dict(zip(seq['pids'], seq['names']))}")
    x, y = seq["x"], seq["y"]
    dx, dy = x[1:] - x[:-1], y[1:] - y[:-1]
    rep = lambda a: a[None].repeat(n, *[1] * a.dim()).requires_grad_(True)  # one copy of the window per frame so every frame gets its own gradient
    inputs = [rep(a) for a in (x[:-1], y[:-1], dx, dy, seq["puck_x"][:-1], seq["puck_y"][:-1])]
    X, Y, DX, DY, PX, PY = inputs
    move, v = model(seq["player_ids"][None].expand(n, -1), seq["player_pos"][None].expand(n, -1), X, Y,
                    seq["off"][None, :-1].expand(n, -1, -1), DX, DY, PX, PY)
    b = torch.arange(n)
    centers = (torch.arange(MOVE_BINS) + 0.5) * 2 * MAX_MOVE / MOVE_BINS - MAX_MOVE  # feet per frame at the middle of each move bin
    p = move[b, b].softmax(-1).detach().reshape(n, K, MOVE_BINS, MOVE_BINS)  # [frame, player, dx bin, dy bin]
    expected = {"exp_dx": (p.sum(3) * centers).sum(2).numpy(), "exp_dy": (p.sum(2) * centers).sum(2).numpy(),  # [frame, player] feet per frame
                "move_probs": p.numpy(), "actual_bins": torch.stack([discretize_move(dx), discretize_move(dy)], -1).numpy()}  # [frame, player, dx bin, dy bin], [frame, player, 2]
    per_entity = lambda g: torch.cat([sum(gi ** 2 for gi in g[:4]).sum(1), (g[4] ** 2 + g[5] ** 2).sum(1)[:, None]], 1).sqrt()  # [frame, entity], puck last
    if target == "epv":  # signed: how much each entity's movement is raising (+) or lowering (-) epv
        gx, gy, gpx, gpy = torch.autograd.grad(v[b, b, 0].sigmoid().sum(), [X, Y, PX, PY])
        step = lambda a: torch.cat([torch.zeros_like(a[:1]), a[1:] - a[:-1]])  # movement over each frame
        players = (gx * step(x[:-1]) + gy * step(y[:-1])).sum(1)
        puck = (gpx * step(seq["puck_x"][:-1]) + gpy * step(seq["puck_y"][:-1])).sum(1)
        return {"infl": torch.cat([players, puck[:, None]], 1).numpy(), "epv": v[b, b, 0].sigmoid().detach().numpy(), **expected}
    actual = discretize_move(dx) * MOVE_BINS + discretize_move(dy)  # [frame, player] move each player actually made
    infl = np.full((n, K), np.nan)
    for k in range(K):
        if k == k_t:
            continue
        f = move[b, b, k].log_softmax(-1)[b, actual[:, k]]
        g = torch.autograd.grad(f.sum(), inputs, retain_graph=True)
        e = per_entity(g)
        e[:, k] = 0  # leave out the player's own inputs
        infl[:, k] = (e[:, k_t] / e.sum(1)).numpy()  # share of player k's movement driven by the target
    return {"infl": infl, "epv": v[b, b, 0].sigmoid().detach().numpy(), **expected}

def draw_rink(ax):
    ax.add_patch(FancyBboxPatch((-100, -42.5), 200, 85, boxstyle="round,pad=0,rounding_size=28", fc="#f4f8fb", ec="black", lw=1.5, zorder=0))
    for xl, c, w in [(0, "red", 2), (-25, "blue", 3), (25, "blue", 3), (-89, "red", 1), (89, "red", 1)]:
        ax.plot([xl, xl], [-42.5, 42.5], c=c, lw=w, zorder=0.5, alpha=0.6)
    for cx, cy in [(0, 0), (-69, 22), (-69, -22), (69, 22), (69, -22)]:
        ax.add_patch(Circle((cx, cy), 15, fill=False, ec="red", lw=1, alpha=0.5, zorder=0.5))
    for gx, th in [(-89, (-90, 90)), (89, (90, 270))]:
        ax.add_patch(Arc((gx, 0), 12, 12, theta1=th[0], theta2=th[1], ec="red", lw=1, zorder=0.5))
    ax.set_xlim(-101, 101)
    ax.set_ylim(-43.5, 43.5)
    ax.set_aspect("equal")
    ax.axis("off")

def draw_move_grid(ax, title):
    im = ax.imshow(np.zeros((MOVE_BINS, MOVE_BINS)), origin="lower", cmap="Greys", vmin=0, vmax=1, extent=(0, MOVE_BINS, 0, MOVE_BINS))
    ax.set_xticks(range(MOVE_BINS + 1))
    ax.set_yticks(range(MOVE_BINS + 1))
    ax.tick_params(length=0, labelbottom=False, labelleft=False)
    ax.grid(c="lightgray", lw=0.6)
    ax.add_patch(Rectangle((MOVE_BINS // 2, MOVE_BINS // 2), 1, 1, fill=False, ec="blue", lw=1.2, zorder=3))  # no movement
    ax.set_title(title, fontsize=10)
    return im

def render(game_id, event_id, target, epv=True, filename=None, draw_arrow=False, grid=False):
    if grid and target in ("puck", "epv"):
        raise ValueError("grid=True needs a player as the target")
    seq = load_shot(game_id, event_id)
    out = run_model(seq, target)
    T = len(seq["x"]) - 1
    x, y, off = seq["x"].numpy(), seq["y"].numpy(), seq["off"].numpy()
    px, py = seq["puck_x"].numpy(), seq["puck_y"].numpy()
    if target == "epv":
        sx, sy, tname = None, None, "EPV"
    elif target == "puck":
        sx, sy, tname = px, py, "puck"
    else:
        k_t = seq["pids"].index(target)
        sx, sy, tname = x[:, k_t], y[:, k_t], seq["names"][k_t]
    vmin, vmax = np.nanpercentile(out["infl"], [5, 95])
    cmap = "Reds"
    if target == "epv":
        vmax = np.percentile(np.abs(out["infl"]), 95)
        vmin, cmap = -vmax, "RdBu_r"
    rel = lambda v: np.clip((v - vmin) / (vmax - vmin), 0, 1)
    ratios = [2.2] + [1] * epv + [1] * grid
    fig = plt.figure(figsize=(11, 5.5 + 2.5 * epv + 2.5 * grid))
    gs = fig.add_gridspec(len(ratios), 1, height_ratios=ratios)
    ax = fig.add_subplot(gs[0])
    if epv:
        ax2 = fig.add_subplot(gs[1])
    if grid:
        grid_gs = gs[-1].subgridspec(1, 2)
        actual_im = draw_move_grid(fig.add_subplot(grid_gs[0]), f"{tname}: actual move")
        pred_im = draw_move_grid(fig.add_subplot(grid_gs[1]), f"{tname}: predicted move")
        target_probs, target_bins = out["move_probs"][:, k_t], out["actual_bins"][:, k_t]
        pred_im.set_clim(0, target_probs.max())
    draw_rink(ax)
    teams = [np.flatnonzero(off[0] > 0.5), np.flatnonzero(off[0] <= 0.5)]
    dots = [ax.scatter(x[0, k], y[0, k], s=170, marker=m, c=np.zeros(len(k)), cmap=cmap, vmin=vmin, vmax=vmax, ec="black", zorder=3)
            for k, m in zip(teams, ["^", "s"])]
    edges = [ax.plot([], [], c="crimson", zorder=2)[0] for _ in range(PLAYERS_FRAME)]
    labels = [ax.text(0, 0, seq["sweater"][k], fontsize=7, zorder=4) for k in range(PLAYERS_FRAME)]
    src = ax.scatter([], [], s=400, facecolors="none", ec="crimson", lw=2, zorder=5)
    if draw_arrow:
        arrow_u, arrow_v = out["exp_dx"] * ARROW_SECONDS * FRAME_HZ, out["exp_dy"] * ARROW_SECONDS * FRAME_HZ  # feet
        arrows = ax.quiver(x[0], y[0], arrow_u[0], arrow_v[0], angles="xy", scale_units="xy", scale=1, minlength=0, color="black", width=0.003, zorder=4)
    #puck_trail, = ax.plot([], [], c="black", lw=1, alpha=0.5, zorder=2)
    if target == "epv":
        puck = ax.scatter([px[0]], [py[0]], s=60, c=[0], cmap=cmap, vmin=vmin, vmax=vmax, ec="black", zorder=6)
    else:
        puck = ax.scatter([px[0]], [py[0]], s=40, c="black", zorder=6)
    fig.colorbar(dots[0], ax=ax, fraction=0.02, pad=0.01, label="EPV change from movement" if target == "epv" else f"influence of {tname}")
    title = ax.set_title("")
    if epv:
        ax2.set_xlim(seq["t"][0], seq["t"][T - 1] + 0.1)
        ax2.set_ylim(0, max(out["epv"].max(), seq["xG"]) * 1.15)
        ax2.axhline(seq["xG"], ls="--", c="gray", lw=1, label=f"shot xG {seq['xG']:.3f}")
        ax2.set_xlabel("seconds to shot")
        ax2.set_ylabel("EPV")
        ax2.legend(loc="upper left")
        line, = ax2.plot([], [], c="tab:orange", lw=2)
        cursor = ax2.axvline(seq["t"][0], c="black", lw=0.8)
    def update(t):
        a = out["infl"][t]
        for d, k in zip(dots, teams):
            d.set_offsets(np.c_[x[t, k], y[t, k]])
            d.set_array(np.where(np.isnan(a[k]), vmin, a[k]))
        for k, e in enumerate(edges if sx is not None else []):
            e.set_data([sx[t], x[t, k]], [sy[t], y[t, k]])
            e.set_linewidth(0 if np.isnan(a[k]) else 0.3 + 4 * rel(a[k]))
            e.set_alpha(0 if np.isnan(a[k]) else 0.1 + 0.9 * rel(a[k]))
        if grid:
            cell = np.zeros((MOVE_BINS, MOVE_BINS))
            cell[target_bins[t, 1], target_bins[t, 0]] = 1
            actual_im.set_data(cell)
            pred_im.set_data(target_probs[t].T)
        if draw_arrow:
            arrows.set_offsets(np.c_[x[t], y[t]])
            arrows.set_UVC(arrow_u[t], arrow_v[t])
        for k, lab in enumerate(labels):
            lab.set_position((x[t, k] + 2, y[t, k] + 2))
        if sx is not None:
            src.set_offsets([[sx[t], sy[t]]])
        puck.set_offsets([[px[t], py[t]]])
        if target == "epv":
            puck.set_array([a[-1]])
        #puck_trail.set_data(px[max(0, t - trail):t + 1], py[max(0, t - trail):t + 1])
        title.set_text("Influence on EPV" if target == "epv" else f"Influence of {tname}")
        if epv:
            line.set_data(seq["t"][:t + 1], out["epv"][:t + 1])
            cursor.set_xdata([seq["t"][t]])
        return []
    fig.tight_layout()
    anim = FuncAnimation(fig, update, frames=T, interval=1000 / FRAME_HZ)
    plt.close(fig)
    if filename is not None:
        anim.save(filename)
    return HTML(anim.to_jshtml())

In [66]:
def game_plays(game_id):
    g = df[df["game_id"] == game_id]
    if g.empty:
        raise ValueError(f"no tracking data for game {game_id}")
    plays = g.groupby("seq_ev_id").first().sort_values(["periodNumber", "secondsElapsedInPeriod"])
    out = pd.DataFrame({"event_id": plays.index,
                        "period": plays["periodNumber"].values,
                        "time": [f"{int(s) // 60}:{int(s) % 60:02d}" for s in plays["secondsElapsedInPeriod"]],
                        "team": np.where(plays["isHome"], plays["home_on_ice_1_team_abbr"], plays["away_on_ice_1_team_abbr"]),
                        "xG": plays["shot_xG"].values.round(3),
                        "goal": plays["is_goal"].astype(bool).values,
                        "frames": g.groupby("seq_ev_id").size()[plays.index].values})
    def renderable(ev):
        try:
            load_shot(game_id, ev)
            return True
        except ValueError:
            return False
    out["renderable"] = [renderable(ev) for ev in out["event_id"]]
    return out
game = 2025021028
ev = 301
game_plays(game)

,event_id,period,time,team,xG,goal,frames,renderable
0,301,1,6:02,BUF,0.064,True,63,True
1,601,2,2:14,WSH,0.084,True,98,True


In [94]:
tgt = 8479359
render(game, ev, target=tgt, epv=False, draw_arrow=True, grid=True)
#render(game, 748, target=8477934, epv=False,filename="EDMvsWPG_JAN8_DRAISATL.gif")

In [72]:
def attention_maps(seq, target):
    """Attention each head pays to every player and the puck, from the target's prediction at each frame: [layer, head, frame, entity], puck last."""
    K, n = PLAYERS_FRAME, model.n_frames
    calls = []  # the transformer's input and mask, once for the movement pass and once for the EPV pass
    hook = model.trans.register_forward_pre_hook(lambda m, args, kwargs: calls.append((args[0], kwargs["mask"])), with_kwargs=True)
    x, y = seq["x"], seq["y"]
    try:
        with torch.no_grad():
            model(seq["player_ids"][None], seq["player_pos"][None], x[None, :-1], y[None, :-1], seq["off"][None, :-1],
                  (x[1:] - x[:-1])[None], (y[1:] - y[:-1])[None], seq["puck_x"][None, :-1], seq["puck_y"][None, :-1])
    finally:
        hook.remove()
    if target == "epv":  # EPV pass: [players per frame | puck per frame], EPV reads the average of the frame's player tokens
        h, mask = calls[1]
        queries = [list(range(t * K, (t + 1) * K)) for t in range(n)]
        owner = np.r_[np.tile(np.arange(K), n), np.full(n, K)]
    else:  # movement pass: per frame [puck, pos_1, look_1, ..., pos_K, look_K], then K start tokens; a player's move is read from its pos token
        h, mask = calls[0]
        F = 2 * K + 1
        k_t = K if target == "puck" else seq["pids"].index(target)
        queries = [[t * F + (0 if k_t == K else 1 + 2 * k_t)] for t in range(n)]
        owner = np.r_[np.tile(np.r_[K, np.repeat(np.arange(K), 2)], n), np.arange(K)]
    maps = []
    with torch.no_grad():
        for layer in model.trans.layers:
            a = layer.norm1(h) if layer.norm_first else h
            _, w = layer.self_attn(a, a, a, attn_mask=mask, need_weights=True, average_attn_weights=False)  # [1, head, query, key]
            w = w[0].numpy()
            maps.append(np.stack([w[:, q].mean(1) for q in queries], 1))  # [head, frame, key]
            h = layer(h, src_mask=mask)
    A = np.stack(maps)
    return np.stack([A[..., owner == e].sum(-1) for e in range(K + 1)], -1)  # each entity's tokens across all frames, summed

def render_heads(game_id, event_id, target, layer, head, filename=None):
    """layer and head count from 1."""
    seq = load_shot(game_id, event_id)
    att = attention_maps(seq, target)
    n_layers, n_heads, T, K = att.shape[0], att.shape[1], att.shape[2], PLAYERS_FRAME
    if not 1 <= layer <= n_layers or not 1 <= head <= n_heads:
        raise ValueError(f"layer must be between 1 and {n_layers}, head between 1 and {n_heads}")
    a = att[layer - 1, head - 1]  # [frame, entity], puck last
    x, y, off = seq["x"].numpy(), seq["y"].numpy(), seq["off"].numpy()
    px, py = seq["puck_x"].numpy(), seq["puck_y"].numpy()
    if target == "epv":
        sx, sy, tname = None, None, "the EPV prediction"
    elif target == "puck":
        sx, sy, tname = px, py, "the puck"
    else:
        k_t = seq["pids"].index(target)
        sx, sy, tname = x[:, k_t], y[:, k_t], f"{seq['names'][k_t]}'s move prediction"
    others = a if target == "epv" else np.delete(a, K if target == "puck" else k_t, axis=-1)
    vmax = np.percentile(others, 99)  # scaled to everyone but the target, whose attention to its own past usually dominates
    pad = 8
    xlim = max(-101, min(x.min(), px.min()) - pad), min(101, max(x.max(), px.max()) + pad)
    ylim = max(-43.5, min(y.min(), py.min()) - pad), min(43.5, max(y.max(), py.max()) + pad)
    teams = [np.flatnonzero(off[0] > 0.5), np.flatnonzero(off[0] <= 0.5)]
    fig, ax = plt.subplots(figsize=(9, 9 * (ylim[1] - ylim[0]) / (xlim[1] - xlim[0]) + 0.6), layout="constrained")
    draw_rink(ax)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    dots = [ax.scatter(x[0, k], y[0, k], s=200, marker=m, c=np.zeros(len(k)), cmap="Reds", vmin=0, vmax=vmax, ec="black", zorder=3)
            for k, m in zip(teams, ["^", "s"])]
    labels = [ax.text(0, 0, seq["sweater"][k], fontsize=8, zorder=4) for k in range(K)]
    puck = ax.scatter([px[0]], [py[0]], s=80, c=[0], cmap="Reds", vmin=0, vmax=vmax, ec="black", zorder=5)
    ring = ax.scatter([], [], s=520, facecolors="none", ec="crimson", lw=2, zorder=6)
    fig.colorbar(dots[0], ax=ax, fraction=0.03, pad=0.01, label="attention")
    title = ax.set_title("")
    def update(t):
        for d, k in zip(dots, teams):
            d.set_offsets(np.c_[x[t, k], y[t, k]])
            d.set_array(a[t, k])
        for k, lab in enumerate(labels):
            lab.set_position((x[t, k] + 1.2, y[t, k] + 1.2))
        puck.set_offsets([[px[t], py[t]]])
        puck.set_array([a[t, K]])
        if sx is not None:
            ring.set_offsets([[sx[t], sy[t]]])
        title.set_text(f"{seq['names'][k_t]}: Layer {layer}, head {head} attention  ({seq['t'][t]:.1f}s to shot)")
        return []
    anim = FuncAnimation(fig, update, frames=T, interval=1000 / FRAME_HZ)
    plt.close(fig)
    if filename is not None:
        anim.save(filename)
    return HTML(anim.to_jshtml())

In [96]:
render_heads(game, ev,  target=tgt, layer=4, head=1)